In [23]:
import sys
import os
import pandas as pd
sys.path.append(os.path.abspath(".."))
from churn.modelo import dividir_datos, construir_modelo
from churn.datos import cargar_datos
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

In [2]:
df = cargar_datos("../data/dataset.csv")

In [3]:
X_train, X_test, y_train, y_test = dividir_datos(df)

In [4]:
num_cols = [
    "rev_Mean",         # Mean monthly revenue (charge amount)
    "mou_Mean",         # Mean number of monthly minutes of use
    "ovrrev_Mean",      # Mean overage revenue
    "comp_dat_Mean",    # Mean number of completed data calls
    "custcare_Mean",    # Mean number of customer care calls
    "phones",           # Number of handsets issued
    "months",           # Total number of months in service
    "eqpdays",          # Number of days (age) of current mobile phone
    "change_mou",       # Percentage change in monthly minutes of use vs previous three month average
    "change_rev",       # Percentage change in monthly revenue vs previous three month average
    "drop_vce_Mean",    # Mean number of dropped (failed) voice calls
    "drop_dat_Mean",    # Mean number of dropped (failed) data calls
    "drop_blk_Mean",    # Mean number of dropped or blocked calls
    "blck_vce_Mean",    # Mean number of blocked (failed) voice calls
    "blck_dat_Mean",    # Mean number of blocked (failed) data calls
    "unan_dat_Mean",    # Mean number of unanswered data calls
    "totmrc_Mean",      # Mean total monthly recurring charge
    "uniqsubs",         # Number of unique subscribers in the household
    "roam_Mean",        # Mean number of roaming calls
    "datovr_Mean",      # Mean revenue of data overage
    "da_Mean",          # Mean number of directory assisted calls
    "recv_sms_Mean",    # Mean number of received sms
    "callfwdv_Mean",    # Mean number of call forwarding calls
    "callwait_Mean",    # Mean number of call waiting calls
    "threeway_Mean",    # Mean number of three way calls
    "hnd_price",        # Current handset price
]

In [5]:
cat_cols = [
    "refurb_new",   # Handset: refurbished or new
    "infobase",     # InfoBase match
    "dualband",     # Dualband
    "asl_flag",     # Account spending limit
    "hnd_webcap",   # Handset web capability
]

In [16]:
def resumir_cv(tit_nombre: str, nombre: str, resultado: dict) -> dict:
    return {
        tit_nombre: nombre,
        "auc_train": resultado["train_score"].mean(),
        "auc_val": resultado["test_score"].mean(),
        "auc_val_std": resultado["test_score"].std(),
    }

filas = []
# pd.DataFrame(filas)

In [7]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [8]:
resultado_sin_limite = cross_validate(
    construir_modelo(num_cols, cat_cols, DecisionTreeClassifier()),
    X_train, y_train,
    cv=cv,
    scoring="roc_auc",
    return_train_score=True,
    error_score="raise"
)

In [17]:
filas.append(resumir_cv("max_depth","None", resultado_sin_limite))

In [11]:
resultado_max_5 = cross_validate(
    construir_modelo(num_cols, cat_cols, DecisionTreeClassifier(max_depth=5)),
    X_train, y_train,
    cv=cv,
    scoring="roc_auc",
    return_train_score=True,
    error_score="raise"
)

In [18]:
filas.append(resumir_cv("max_depth","5", resultado_max_5))

En la siguiente tabla se muestra como al no poner un límite de profundidad para el árbol de decisión, el modelo hace overfitting: El desempeño en train es perfecto (1.0) mientras que en validación es mucho más bajo (0.55), incluso peor que cuando se usó la regla de eqpdays (0.57). El modelo memoriza el ruido de train

Por otro lado, cuando se le agrega una profundidad máxima de 5, aunque supera lo que se obtuvo con Regresión Logística (tran = val = 0.61), sigue estando con uderfittng: el desempeño de train es bajo y la validación es también baja, parecido a train. El modelo es demasiado simple para el patrón

In [19]:
pd.DataFrame(filas)

,max_depth,auc_train,auc_val,auc_val_std
0,None,1.000000,0.547461,0.005353
1,5,0.639956,0.634456,0.002445


In [15]:
resultado_rand_forest_clf = cross_validate(
    construir_modelo(
        num_cols,cat_cols,
        RandomForestClassifier(
            n_estimators=200,
            min_samples_leaf=50,
            n_jobs=-1,
            random_state=42
        )
    ),
    X_train, y_train,
    cv=cv,
    scoring="roc_auc",
    return_train_score=True,
    error_score="raise"
)

In [34]:
filas.clear()
filas.append(resumir_cv("modelo","RandomForestClassifier", resultado_rand_forest_clf))

In [30]:
resultado_lgbm_clf_imp = cross_validate(
    construir_modelo(
        num_cols,cat_cols,
        LGBMClassifier(
            random_state=42,
            verbose=-1
        )
    ),
    X_train, y_train,
    cv=cv,
    scoring="roc_auc",
    return_train_score=True,
    error_score="raise"
)

In [35]:
filas.append(resumir_cv("modelo","LGBMClassifier (imputando)", resultado_lgbm_clf_imp))

In [ ]:
resultado_lgbm_clf = cross_validate(
    construir_modelo(
        num_cols,cat_cols,
        LGBMClassifier(
            random_state=42,
            verbose=-1
        ),
        False
    ),
    X_train, y_train,
    cv=cv,
    scoring="roc_auc",
    return_train_score=True,
    error_score="raise"
)

In [36]:
filas.append(resumir_cv("modelo","LGBMClassifier (sin impotar)", resultado_lgbm_clf))

En la tabla de abajo se puede observar que el modelo que mejor desempeño tuvo fue LGBMClassifier sin imputar ni escalar variables numéricas. No mostró ni sobreaujuste ni subajuste: su val salió ~5 puntos debajo de train, lo cual no es suficiente para decir que hay sobreajuste.

Aquí quise hacer dos comparaciones para el mismo modelo de LGBMClassifier, una imputando y escalando y otra sin hacerlo. Observé que se comportó mejor cuando no se hace. LightGBM es una implementación de Gradient Boosting que además acepta NaN de forma nativa y no necesita escalado por lo que al hacerlo, se pierden los datos originales y el modelo se comportó peor.Por el contrario, RandomForest sí necesita imputación porque su algoritmo utiliza variaciones del algoritmo CART, el cual asume que los datos están completos. 

LightLGB, es un algoritmo basado en histogramas, diseñado para manejar datos dispersos y valores nulos de forma nativa.
Durante la fase de entrenamiento, cuando encuentra valores nulos en una variable, no falla. En su lugar, prueba temporalmente dos escenarios:
1. Enviar los nulos a la rama **izquierda**.
1. Enviar los nulos a la rama **derecha**.

El algoritmo calcula cuál de las dos opciones reduce más el error del modelo. Cuando el modelo ya está entrenado y recibe datos faltantes, direcciona esos datos hacia el lado por defecto que aprendió.

In [37]:
pd.DataFrame(filas)

,modelo,auc_train,auc_val,auc_val_std
0,RandomForestClassifier,0.738740,0.666932,0.003991
1,LGBMClassifier (imputando),0.737532,0.679870,0.002688
2,LGBMClassifier (sin impotar),0.738928,0.680010,0.002185


### Pruebas de hiperparámetros

In [53]:
cfg_1 = LGBMClassifier()

cfg_2 = LGBMClassifier(
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=15,
    min_child_samples=100,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8
)

cfg_3 = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.1,
    num_leaves=63
)

cfg_4 = LGBMClassifier(
    random_state=42,
    verbose=-1
)

In [38]:
config_regularizada = LGBMClassifier(
    learning_rate= 0.05,   # Aprendizaje más lento
    n_estimators= 300, 
    num_leaves= 15,        # Árboles más "delgados" (menos nodos finales)
    max_depth= 5,          # Profundidad estricta
    min_child_samples= 50, # Exige al menos 50 datos para crear una nueva hoja
    subsample= 0.8,        # Usa solo el 80% de las filas por árbol (Bagging)
    colsample_bytree= 0.8, # Usa solo el 80% de las columnas por árbol
    reg_alpha= 0.1,        # Penalización L1 (Lasso)
    reg_lambda= 0.5,       # Penalización L2 (Ridge)
    random_state= 42
)

In [39]:
config_profunda = LGBMClassifier(
    learning_rate= 0.01,   # Aprendizaje muy lento
    n_estimators= 1000,    # Compensa el aprendizaje lento con muchos árboles
    num_leaves= 63,        # Permite árboles muy complejos
    max_depth= 8,          # Ligeramente profundo
    subsample= 0.9,
    colsample_bytree= 0.9,
    random_state= 42
)

In [40]:
config_dart = LGBMClassifier(
    boosting_type= 'dart', # Cambio radical en la construcción del ensamble
    learning_rate= 0.1,
    n_estimators= 200,
    drop_rate= 0.1,        # Probabilidad de apagar un árbol
    skip_drop= 0.5,        # Probabilidad de saltarse el proceso de dropout en una iteración
    max_drop= 50,          # Máximo de árboles apagados a la vez
    random_state= 42
)

In [ ]:
configuraciones = {
    'default':      cfg_1, 
    'cfg_2':        cfg_2, 
    'cfg_3':        cfg_3, 
    'regularizada': config_regularizada, 
    'profunda':     config_profunda, 
    'cart':         config_dart
}

In [45]:
def resumir_cv(nombre: str, resultado: dict) -> dict:
    train = resultado["train_score"].mean()
    validacion = resultado["test_score"].mean()
    brecha = abs(train - validacion)
    return {
        'configuración': nombre,
        "train": train,
        "val": validacion,
        "brecha": brecha,
    }
# pd.DataFrame(filas)

In [ ]:
filas.clear()

for cfg, estimador in configuraciones.items():
    resultado = cross_validate(
        construir_modelo(num_cols, cat_cols,estimador,False),
        X_train, y_train,
        cv=cv,
        scoring="roc_auc",
        return_train_score=True,
        error_score="raise"
    )
    filas.append(resumir_cv(cfg, resultado))

In [54]:
resultado = cross_validate(
        construir_modelo(num_cols, cat_cols,cfg_4,False),
        X_train, y_train,
        cv=cv,
        scoring="roc_auc",
        return_train_score=True,
        error_score="raise"
    )
filas.append(resumir_cv("cfg_4", resultado))

In [55]:
pd.DataFrame(filas)

,configuración,train,val,brecha
0,default,0.738928,0.680010,0.058918
1,cfg_2,0.728851,0.682635,0.046216
2,cfg_3,0.987739,0.661611,0.326128
3,regularizada,0.715042,0.680393,0.034650
4,profunda,0.778020,0.682799,0.095221
5,cart,0.714886,0.678919,0.035967
6,cart,0.738928,0.680010,0.058918
7,cfg_4,0.738928,0.680010,0.058918


Se puede observar que de estas configuraciones la que más sobrejusta es cfg_3. Se nota porque el train es muy alto (0.99) y el val muy bajo (0.66).

Entre estos 6 modelos, el mejor es el de configuración regularizada. Esta configuración está pensada para restringir la complejidad del árbol y fuerza al modelo a aprender patrones más generales. Se puede ver que los valores de train y val no distan mucho uno del otro, apenas 3.4 puntos porcentuales siendo la brecha más corta, demostrando así que no hay riesgo de overfitting y superando aregresión logística en val (61%) por ~7 puntos.